# Training on a public dataset: bev-decision-150K

> **⚠️ Plan for a long run.** The LoRA and QLoRA sections train the backbone on the full training
> data, and nothing can be cached when the backbone learns, so each one takes far longer than
> head-only training. As a rough guide, Qwen3-0.6B takes about an hour per epoch on a Mac, so up
> to about two hours per section with the defaults, and a fraction of that on an A100. Use a CUDA
> GPU (on Colab, an L4 or A100 runtime) and save trained models to Google Drive with
> `model.save(...)`, since Colab deletes `/content` when the runtime disconnects. A run stopped
> with Ctrl-C keeps its best weights so far and continues from where it stopped when `fit` is
> called again with the same data.

[avbiswas/bev-decision-150K](https://huggingface.co/datasets/avbiswas/bev-decision-150K) holds about
150,000 texts from 18 domains (spatial reasoning, sentiment, retail, support routing, finance, ...),
each with one or more typed questions and their correct answers. It is much harder than the
synthetic tickets in `quickstart.ipynb`: the questions differ from row to row, and many need real
reading comprehension.

This notebook trains Qwen3-0.6B with `tuning="head"` (frozen backbone, only the decision head
learns), then fine-tunes the backbone with `tuning="lora"` and `tuning="qlora"`, and compares all
three with random guessing on held-out test rows, per question type and per domain.
[`encoder_bev_decision.ipynb`](encoder_bev_decision.ipynb) is the same notebook with ModernBERT, a
bidirectional encoder.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.5"

In [ ]:
import json

from datasets import load_dataset

from sysone import DecoderModel, baselines, compare

N_TRAIN = 3000   # training rows (about 2 questions each); the full train split has 125,614
N_TEST = 1000    # held-out rows from the test split
BACKBONE_EPOCHS = 2  # upper bound for LoRA and QLoRA; early stopping usually ends sooner

## Load the data

The correct answer is stored as a `label` inside each question, which sysone reads directly. Each
row's domain is passed as its `category`, so validation is split evenly across domains and
results can be broken down by domain.

In [ ]:
def load_rows(split, n):
    ds = load_dataset("avbiswas/bev-decision-150K", split=split).shuffle(seed=0).select(range(n))
    return [{"state": r["state"], "questions": json.loads(r["questions_json"]), "category": r["domain"]}
            for r in ds]


train = load_rows("train", N_TRAIN)
test = load_rows("test", N_TEST)
print("%d training rows, %d test rows" % (len(train), len(test)))
next(iter(train[0]["questions"].values()))

## Before training: random guessing

Questions differ from row to row here, so the only meaningful baseline is random guessing: a
question with k options is right 1/k of the time.

In [ ]:
results = baselines(test)
compare(results)

## Train

The backbone reads every training question once and its outputs are cached; each epoch then
trains only the head. 10% of rows are held back for validation and early stopping.

In [ ]:
model = DecoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["head"] = model.evaluate(test)
compare(results)

## Fine-tuning the backbone with LoRA

With `tuning="lora"`, small adapters on the top 60% of the kept layers train along with the head,
so the backbone itself adapts to these questions. Nothing can be cached, since the backbone's
outputs change every step: each example costs a forward and a backward pass through the backbone
every epoch. It trains on the same questions as the head-only model above, so the two compare
directly.

Validation runs about ten times per epoch, and training stops early once it stops improving, so
`BACKBONE_EPOCHS` is an upper bound.

In [ ]:
lora = DecoderModel(tuning="lora", epochs=BACKBONE_EPOCHS)
lora.fit(train)
results["lora"] = lora.evaluate(test)
compare(results)

## Fine-tuning the backbone with QLoRA

`tuning="qlora"` (the default) trains the same adapters as LoRA, but stores the backbone's frozen
weights in 4-bit, so it needs much less GPU memory. It is somewhat slower per step, because the
weights are unpacked on the fly, and accuracy is usually close to LoRA's.

4-bit loading needs bitsandbytes and an NVIDIA GPU (any Colab GPU runtime). Elsewhere, for
example on a Mac, QLoRA falls back to LoRA with a warning and trains exactly like the section
above, so skip this section there. The cell prints which mode it actually used.

In [ ]:
qlora = DecoderModel(tuning="qlora", epochs=BACKBONE_EPOCHS)
qlora.fit(train)
print("tuning used:", qlora.tuning_)  # "lora" means 4-bit wasn't available and it fell back
memory = qlora.backbone_.get_memory_footprint() / 1e9, lora.backbone_.get_memory_footprint() / 1e9
print("backbone memory: %.2f GB with QLoRA, %.2f GB with LoRA" % memory)
results["qlora"] = qlora.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).
- **brier**: squared distance between the predicted probabilities and the label; lower is better.
- **ece**: expected calibration error, how far "says X% sure" is from "right X% of the time";
  lower is better.

Probabilities are calibrated after training: a temperature per question type, fitted on the
validation slice, softens over-confident answers. This changes **log_loss**, **brier** and **ece**,
never **accuracy**, since the top answer stays the same.

In [ ]:
compare(results, by="overall")

### Calibration

The same model before and after temperature scaling. Accuracy is identical; the probabilities, and
so **log_loss**, **brier** and **ece**, change. `lora.report_["calibration"]` has the fitted
temperatures and the validation slice's ECE before and after.

In [ ]:
calibration = {"lora": results["lora"], "lora, before calibration": lora.evaluate(test, calibrated=False)}
compare(calibration, by="overall")[["accuracy", "log_loss", "brier", "ece"]]

### Accuracy by domain

The number of test questions per domain is in `results["head"]["by_category"][domain]["n"]`;
small domains give noisy numbers.

In [ ]:
compare(results, by="category").drop(columns="overall").T.sort_values("head", ascending=False)

## What to expect

Training only the head on a frozen backbone is the cheapest setting and the least accurate: the
backbone was never adapted to these questions. Expect clear gains over random guessing, and LoRA and
QLoRA to do better than head-only on the same rows. QLoRA should land close to LoRA; a gap of a
point or two is within run-to-run noise. Score questions are the hardest. For them the loss adds an
ordinal term (`ordinal_weight`), so predicting a level next to the right one costs less than one far
away; **score_error** shows how far off the predicted level is on average.

Things to try: more training rows (`N_TRAIN`), more `BACKBONE_EPOCHS`, a different `num_layers`
(default 0.7 of Qwen's 28 layers), or `lora_targets="all"`. A long run stopped with Ctrl-C continues
from where it stopped with `lora.fit(train)`.